In [1]:
!pip -q install -U datasets transformers peft trl accelerate bitsandbytes huggingface_hub

In [2]:
!rm -rf ./data/qa_abstain_dataset_clipped
!rm -f ./data/abstain_train.jsonl
!rm -rf ./outputs_abstain_fast
!rm -rf ./outputs_abstain
!rm -rf ./adapter_abstain*

In [3]:
from huggingface_hub import login
# login()   # uncomment to login

In [ ]:
import os, json, re, random
import torch

from datasets import load_dataset, load_from_disk
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

ABSTAIN_STR = "FINAL: I don't know."

DATA_DIR = "./data"
JSONL_PATH = os.path.join(DATA_DIR, "abstain_train.jsonl")
CACHE_DIR = os.path.join(DATA_DIR, "qa_abstain_dataset_clipped")

# Prevent RAM blowups: keep only first N chars of each context
# MAX_CHARS = 2000  # reduce to 1200 if you still crash later / want faster training

# Dataset sizes (tune as needed)
N_TRIVIA_TARGET = 15000   # realistic on streaming filter; bump if you want more
N_UNANS_TARGET  = 7000

os.makedirs(DATA_DIR, exist_ok=True)

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip().lower())

# def clip(text: str, max_chars=MAX_CHARS) -> str:
#     t = (text or "").strip()
#     return t[:max_chars]

# Prevent RAM blowups
MAX_CHARS = 2000
MIN_WINDOW = 800
JITTER = 150

def clip(text: str, max_chars=MAX_CHARS) -> str:
    """
    Generic clipping for unanswerable contexts:
    if too long, take a random window so lengths look similar
    to answerable examples.
    """
    text = (text or "").strip()
    if len(text) <= max_chars:
        return text
    start = random.randint(0, max(0, len(text) - max_chars))
    return text[start:start + max_chars].strip()


def find_best_match(context: str, answers):
    """
    Returns (start, end) for the first answer alias found in context.
    """
    if isinstance(answers, str):
        answers = [answers]

    ctx_lower = context.lower()
    for ans in answers:
        if not ans or not ans.strip():
            continue
        ans_clean = ans.strip()
        idx = ctx_lower.find(ans_clean.lower())
        if idx != -1:
            return idx, idx + len(ans_clean)
    return None


def adjust_left_boundary(text: str) -> str:
    candidates = []
    for pat in [r"\.\s", r"\?\s", r"!\s", r"\n"]:
        m = re.search(pat, text)
        if m:
            candidates.append(m.end())
    if candidates:
        cut = min(candidates)
        if cut < 300:
            return text[cut:]
    return text


def adjust_right_boundary(text: str) -> str:
    candidates = []
    for pat in [r"\.\s", r"\?\s", r"!\s", r"\n"]:
        matches = list(re.finditer(pat, text))
        if matches:
            candidates.append(matches[-1].end())
    if candidates:
        cut = max(candidates)
        if len(text) - cut < 300:
            return text[:cut]
    return text


def shrink_context_around_answer(
    context: str,
    answers,
    max_chars: int = MAX_CHARS,
    min_window: int = MIN_WINDOW,
    jitter: int = JITTER,
):
    """
    Clip context around a found answer span.
    Returns None if no answer alias is found.
    """
    if not context:
        return None

    match = find_best_match(context, answers)
    if match is None:
        return None

    start_ans, end_ans = match

    if len(context) <= max_chars:
        return context.strip()

    center = (start_ans + end_ans) // 2

    if jitter > 0:
        center += random.randint(-jitter, jitter)
        center = max(0, min(center, len(context) - 1))

    half = max_chars // 2
    start = max(0, center - half)
    end = min(len(context), center + half)

    snippet = context[start:end]

    if start > 0:
        snippet = adjust_left_boundary(snippet)
    if end < len(context):
        snippet = adjust_right_boundary(snippet)

    if len(snippet) < min_window and len(context) > min_window:
        half2 = min_window // 2
        start2 = max(0, center - half2)
        end2 = min(len(context), center + half2)
        snippet = context[start2:end2]

    return snippet.strip()

In [ ]:
INSTR = (
  "Instruction: Answer using ONLY the context. "
  "If the answer is not in the context, reply exactly with: I don't know.\n\n"
)

def build_jsonl_dataset(jsonl_path: str):
    if os.path.exists(jsonl_path):
        os.remove(jsonl_path)

    def format_trivia_stream(example):
        q = example["question"]
        a = example["answer"]["value"].strip()
        a_norm = normalize(a)

        aliases = example["answer"].get("aliases", []) or []
        aliases = [x.strip() for x in aliases if x and x.strip()]
        if a not in aliases:
            aliases = [a] + aliases

        contexts = example["entity_pages"]["wiki_context"] or []
        chosen = None
        for ctx in contexts:
            if any(normalize(alias) in normalize(ctx) for alias in aliases):
                chosen = ctx
                break
        if not chosen:
            return None

        chosen = shrink_context_around_answer(
            context=chosen,
            answers=aliases,
            max_chars=MAX_CHARS,
            min_window=MIN_WINDOW,
            jitter=JITTER,
        )

        if not chosen:
            return None

        text = INSTR + f"Context: {chosen}\nQuestion: {q}\nAnswer:\nFINAL: {a}"
        return {"text": text, "label": "answerable"}

    def format_squad_unanswerable_stream(example):
        if len(example["answers"]["text"]) != 0:
            return None

        ctx = clip(example["context"])
        q = example["question"].strip()

        text = INSTR + f"Context: {ctx}\nQuestion: {q}\nAnswer:\n{ABSTAIN_STR}"
        return {"text": text, "label": "unanswerable"}

    count_ans = 0
    count_unans = 0

    with open(jsonl_path, "w") as f:
        print("Streaming TriviaQA...")
        trivia_stream = load_dataset("mandarjoshi/trivia_qa", "rc", split="train", streaming=True)

        for ex in trivia_stream:
            item = format_trivia_stream(ex)
            if item is None:
                continue
            f.write(json.dumps(item) + "\n")
            count_ans += 1
            if count_ans >= N_TRIVIA_TARGET:
                break

        print("Streaming SQuAD 2.0 (unanswerables)...")
        squad_stream = load_dataset("rajpurkar/squad_v2", split="train", streaming=True)

        for ex in squad_stream:
            item = format_squad_unanswerable_stream(ex)
            if item is None:
                continue
            f.write(json.dumps(item) + "\n")
            count_unans += 1
            if count_unans >= N_UNANS_TARGET:
                break

    print(f"Wrote JSONL: {jsonl_path}")
    print(f"Answerable: {count_ans}, Unanswerable: {count_unans}, Total: {count_ans + count_unans}")

In [6]:
def get_gold_from_text(text: str) -> str:
    # gold is always the last line like "FINAL: ..."
    last_line = text.strip().splitlines()[-1].strip()
    return last_line  # includes "FINAL: ..."

In [7]:
from datasets import load_dataset as hf_load_dataset

def build_cache_from_jsonl(jsonl_path: str, cache_dir: str):
    ds = hf_load_dataset("json", data_files=jsonl_path, split="train")
    split = ds.train_test_split(test_size=0.05, seed=SEED)

    if os.path.exists(cache_dir):
        import shutil
        shutil.rmtree(cache_dir)

    split.save_to_disk(cache_dir)
    print("Saved dataset cache:", cache_dir)
    print(split)

In [8]:
# Build JSONL then cache
build_jsonl_dataset(JSONL_PATH)
build_cache_from_jsonl(JSONL_PATH, CACHE_DIR)

Streaming TriviaQA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Streaming SQuAD 2.0 (unanswerables)...
Wrote JSONL: ./data/abstain_train.jsonl
Answerable: 15000, Unanswerable: 7000, Total: 22000


Generating train split: 0 examples [00:00, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20900 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1100 [00:00<?, ? examples/s]

Saved dataset cache: ./data/qa_abstain_dataset_clipped
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20900
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1100
    })
})


In [9]:
# Safe default (open + small)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# If you have access and want bigger:
# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
# MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"  # requires access
print("Using model:", MODEL_NAME)

Using model: Qwen/Qwen2.5-1.5B-Instruct


In [10]:
ds = load_from_disk(CACHE_DIR)
train_ds = ds["train"]
eval_ds  = ds["test"]

print("Train:", len(train_ds), "Eval:", len(eval_ds))
print("Sample:", train_ds[0])
print(train_ds[0]["text"][:250])  # should show Instruction + Answer: + FINAL:

Train: 20900 Eval: 1100
Sample: {'text': 'Instruction: Answer using ONLY the context. If the answer is not in the context, reply exactly with: I don\'t know.\n\nContext: The monarchy of the United Kingdom, commonly referred to as the British monarchy, is the constitutional monarchy of the United Kingdom and its overseas territories. The monarch\'s title is "King" (male) or "Queen" (female). The current monarch and head of state, Queen Elizabeth II, ascended the throne on the death of her father, King George VI, on 6\xa0February 1952.\n\nThe monarch and his or her immediate family undertake various official, ceremonial, diplomatic and representational duties. As the monarchy is constitutional, the monarch is limited to non-partisan functions such as bestowing honours and appointing the Prime Minister. The monarch is, by tradition, commander-in-chief of the British Armed Forces. Though the ultimate formal executive authority over the government of the United Kingdom is still by and throu

In [11]:
ex = train_ds[0]
gold_line = get_gold_from_text(ex["text"])
gold = gold_line.replace("FINAL:", "", 1).strip()

ctx = ex["text"].split("Context:",1)[1].split("Question:",1)[0]
print("Gold:", gold)
print("Gold in context?", gold.lower() in ctx.lower())

Gold: King George VI
Gold in context? True


In [12]:
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    !nvidia-smi

# Try QLoRA (4-bit). If it fails, we fall back to normal LoRA.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
    )
    print("Loaded model in 4-bit (QLoRA-ready).")
    quant_mode = "4bit"
except Exception as e:
    print("4-bit load failed, falling back to full precision LoRA.\nReason:", repr(e))
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16 if use_cuda else torch.float32,
    )
    quant_mode = "fp16/fp32"

CUDA available: True
Wed Mar  4 23:45:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded model in 4-bit (QLoRA-ready).


In [13]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [16]:
def split_text_to_prompt_resp(text: str):
    lines = text.strip().splitlines()
    gold_resp = lines[-1].strip()
    prompt = "\n".join(lines[:-1]).strip() + "\n"
    return prompt, gold_resp

def extract_final_answer(s: str) -> str:
    m = re.search(r"FINAL:\s*(.*)", s, flags=re.IGNORECASE)
    return m.group(1).strip() if m else s.strip()

In [17]:
print(train_ds[0]["text"][:250])

Instruction: Answer using ONLY the context. If the answer is not in the context, reply exactly with: I don't know.

Context: The monarchy of the United Kingdom, commonly referred to as the British monarchy, is the constitutional monarchy of the Unite


In [18]:
p, g = split_text_to_prompt_resp(eval_ds[0]["text"])
print("PROMPT tail:", p[-200:])
print("GOLD:", g)

PROMPT tail: rict squire, Trelawney, proposes buying a ship and going after the treasure, taking Livesey as ship's doctor and
Question: In ‘Treasure Island’, what is the name of Long John Silver’s parrot?
Answer:

GOLD: FINAL: CAPTAIN FLINT


In [19]:
prompt, gold_resp = split_text_to_prompt_resp(eval_ds[0]["text"])
gold = extract_final_answer(gold_resp)

# extract context from the prompt
ctx = prompt.split("Context:",1)[1].split("Question:",1)[0]
print("Gold:", gold)
print("Gold in context?", gold.lower() in ctx.lower())

Gold: CAPTAIN FLINT
Gold in context? True


In [ ]:


from trl import SFTTrainer, SFTConfig
from datasets import concatenate_datasets
import torch

# --- small subset ---
# from datasets import concatenate_datasets

ans = train_ds.filter(lambda x: x["label"]=="answerable").shuffle(seed=42).select(range(4750))
unans = train_ds.filter(lambda x: x["label"]=="unanswerable").shuffle(seed=42).select(range(250))
train_small = concatenate_datasets([ans, unans]).shuffle(seed=42)
eval_small = eval_ds.shuffle(seed=42).select(range(200))

# --- stability ---
model.gradient_checkpointing_disable()
model.config.use_cache = True

training_args = SFTConfig(
    output_dir="./outputs_abstain_10min",
    max_length=192,                 # smaller = faster
    packing=False,

    per_device_train_batch_size=4,  # if OOM, set to 2
    gradient_accumulation_steps=4,  # effective batch 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,

    num_train_epochs=1,
    max_steps=100,                  # HARD CAP => ~10 min target

    eval_strategy="no",
    save_steps=100,
    logging_steps=10,
    save_total_limit=1,

    gradient_checkpointing=False,
    bf16=torch.cuda.is_available(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_small,
    eval_dataset=eval_small,
    args=training_args,
)

trainer.train()

Filter:   0%|          | 0/20900 [00:00<?, ? examples/s]

Filter:   0%|          | 0/20900 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.243182
20,1.849497
30,1.691955
40,1.678694
50,1.650924
60,1.654020
70,1.644105
80,1.627941
90,1.574152
100,1.667735


TrainOutput(global_step=100, training_loss=1.7282205390930176, metrics={'train_runtime': 1042.2632, 'train_samples_per_second': 1.535, 'train_steps_per_second': 0.096, 'total_flos': 2449254069043200.0, 'train_loss': 1.7282205390930176})

In [21]:
ADAPTER_DIR = "./adapter_abstain"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to:", ADAPTER_DIR)
print("Quantization mode used:", quant_mode)

Saved adapter to: ./adapter_abstain
Quantization mode used: 4bit


In [22]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Reload base (same as before)
try:
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
    )
except Exception:
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16 if use_cuda else torch.float32,
    )

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
print("Reloaded base + adapter.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Reloaded base + adapter.


In [ ]:
import re, string
import torch

@torch.no_grad()
def generate_tail(prompt: str, max_new_tokens=32):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # greedy
        temperature=1.0,
        pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id),
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

def is_abstain(s: str) -> bool:
    return normalize(s).startswith(normalize(ABSTAIN_STR))

def norm(s: str) -> str:
    s = (s or "").lower().strip()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s)
    return s

def relaxed_match(pred: str, gold: str) -> bool:
    p, g = norm(pred), norm(gold)
    return p == g or p in g or g in p



sample_n = min(400, len(eval_ds))
sample = eval_ds.shuffle(seed=42).select(range(sample_n))

tp_ans = ans_total = 0
refuse_on_unans = unans_total = 0
over_refuse = hallucinate = 0

for ex in sample:
    prompt, gold_resp = split_text_to_prompt_resp(ex["text"])
    label = ex["label"]

    pred_resp = generate_tail(prompt)
    pred_is_abstain = is_abstain(pred_resp)

    if label == "answerable":
        ans_total += 1
        if pred_is_abstain:
            over_refuse += 1
        else:
            pred_final = extract_final_answer(pred_resp)
            gold_final = extract_final_answer(gold_resp)
            if relaxed_match(pred_final, gold_final):
                tp_ans += 1
    else:
        unans_total += 1
        if pred_is_abstain:
            refuse_on_unans += 1
        else:
            hallucinate += 1

print("\n==== Metrics (quick eval) ====")
print(f"Accuracy (answerable):        {tp_ans / max(1, ans_total):.3f}")
print(f"Refusal rate (unanswerable):  {refuse_on_unans / max(1, unans_total):.3f}")
print(f"Over-refusal (answerable):    {over_refuse / max(1, ans_total):.3f}")
print(f"Hallucination proxy (unans):  {hallucinate / max(1, unans_total):.3f}")
print(f"Eval samples used: {sample_n}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
prompt, gold_resp = split_text_to_prompt_resp(eval_ds[0]["text"])
pred_tail = generate_tail(prompt)
print("PRED TAIL:\n", pred_tail)
print("GOLD:\n", gold_resp)

In [ ]:
for i in range(5):
    ex = sample[i]
    prompt, gold_resp = split_text_to_prompt_resp(ex["text"])
    pred = generate_tail(prompt)
    print("\nLABEL:", ex["label"])
    print("PROMPT:\n", prompt[:300], "...")
    print("PRED:\n", pred)
    print("GOLD:\n", gold_resp)